In [ ]:
#use scikit-learn to grid search best hyper-parameters for the Balanced Random Forest Classifier Model
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime
#import config

import pickle
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn import impute
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, make_scorer, fbeta_score, accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

"""
    use GridSearchCV to search for best hyper-parameters, and use 4-fold cross validaiton for parameter search
    :param df: full dataset, do not apply train_test_split here
    :param feature_names: feature names
    :param f5_socres: a self-defined evaluation metrics, similar to recall score
    :return:

"""

#df = pd.read_excel(r'structed_features_2020_12_31_public.xlsx', engine='openpyxl')
#df = pd.read_excel(r'structed_features_2020_12_31_private.xlsx', engine='openpyxl')
#df = pd.read_csv(r'structed_features_2020_12_31_public.csv', encoding='ANSI')
df = pd.read_csv(r'structed_features_2020_12_31_private.csv', encoding='ANSI')


df = df.rename(columns={name: name.lower() for name in df})
feature_names = [col for col in df if col.startswith('m') or col.startswith('f')]
y = df['isbad'].values

# data pre-processing
df['m007'][np.isnan(df['m007'])] = 0
df['m009'][np.isnan(df['m009'])] = 0
df['m011'][np.isnan(df['m011'])] = 0 # only use for public, not private!
df['m012'][np.isnan(df['m012'])] = 0
df['m013'][np.isnan(df['m013'])] = 0
df['m016'][np.isnan(df['m016'])] = 0

df['m014'][np.isnan(df['m014'])] = 1
df['m015'][np.isnan(df['m015'])] = 1

df['m008'][np.isnan(df['m008'])] = 90
df['m010'][np.isnan(df['m010'])] = 90


X = df[feature_names].values

simple_imputer = impute.SimpleImputer(strategy='mean')
X = simple_imputer.fit_transform(X)

#define hyper-parameter options
n_estimators = [80, 100, 200]
max_depth = [None, 1, 3]
min_samples_split = [2, 4, 8, 12]
min_samples_leaf = [0.005, 0.01, 0.1, 0.3]
min_weight_fraction_leaf = [0, 0.01]
oob_score = [True, False]

param_grid = dict(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
                 min_weight_fraction_leaf=min_weight_fraction_leaf, oob_score=oob_score)

cv = 4
scoring_method = 'f5'
f5_score = make_scorer(fbeta_score, beta=5)

print('dataset and model selection')
print('total samples', len(df))
print('total positive samples', y.sum())
print('cross validation number', cv)
print('hyper-parameters combinations', np.prod([len(param_grid[key]) for key in param_grid]).item())
print('scoring method', scoring_method)

print('expected searched hyper-parameters')
print(param_grid)


clf = BalancedRandomForestClassifier(random_state=1)
gs = GridSearchCV(
    clf,
    param_grid=param_grid,
    scoring=f5_score,
    cv=cv,
    return_train_score=True,
    verbose=1)
gs.fit(X, y)
print('the best hyper-parameters')
print(gs.best_params_)
print('the best evaluation score')
print(f'f5={gs.best_score_: .4f}')


metrics_df = pd.DataFrame({
    'train_score_mean': gs.cv_results_['mean_train_score'],
    'train_score_std': gs.cv_results_['std_train_score'],
    'fit_time_mean': gs.cv_results_['mean_fit_time'],
    'fit_time_std': gs.cv_results_['std_fit_time'],
    'test_score_mean': gs.cv_results_['mean_test_score'],
    'test_score_std': gs.cv_results_['std_test_score'],
    'score_time_mean': gs.cv_results_['mean_score_time'],
    'score_time_std': gs.cv_results_['std_score_time'],
    'hyper_params': gs.cv_results_['params']
})
print('Hyper-parameters search details')
print(metrics_df)

In [ ]:
#check the best estimator evaluation metrics including Recall/Precision/F1/AUC scores#######
def _get_score(estimator, X, y):
    y_pred = estimator.predict(X)
    pos_label_idx = np.where(estimator.classes_==1)[0][0]
    y_pred_proba = estimator.predict_proba(X)[:, pos_label_idx]
    return recall_score(y, y_pred), precision_score(y, y_pred), f1_score(y, y_pred), roc_auc_score(y, y_pred_proba)

data = [
    ['cross validation full dataset outcome'] + list(_get_score(gs.best_estimator_, X, y))
]

clf = gs.best_estimator_

print('again re-using full dataset training model')
clf.fit(X, y)
data.append(['after training using full dataset'] + list(_get_score(clf, X, y)))
print('Evaluation of training results')
evaluation_df = pd.DataFrame(data, columns=['', 'Recall', 'Precision', 'F1', 'AUC'])
print(evaluation_df)

In [ ]:
#save final optimised model with best hyper-parameters 
#model_path = 'model1.pkl'
model_path = 'model2.pkl'
with open(model_path, 'wb')  as f:
    pickle.dump(clf, f)